# FAST-001 — three-arm low-cost CLM exploration

Default **Run All performs CPU preflight only**. No GPU training is approved by this Notebook's defaults. Use the existing `bingjiangli/minicells` Notebook. Historical experiments, checkpoints and confirmation data remain untouched.

After reviewing engineering results, explicitly choose `MODE='benchmark'` and `ALLOW_GPU_PILOT=True` on one otherwise idle T4. All three models are measured serially with microbatch 1 and 2; the pilot does not launch training. Save Version to retain Output. Then review the measured total-time projection before choosing development training.


In [ ]:
import json
import time
from pathlib import Path
import subprocess
import sys
import os
import importlib.util

SESSION_STARTED = time.perf_counter()
MODE = "preflight"  # preflight | benchmark | train-matrix | report
ALLOW_GPU_PILOT = False
APPROVE_DEVELOPMENT = False
ACCEPT_SLOWER_THAN_TARGET = False
RESUME = False
RESTORE_FROM = ""  # optional user-attached retained portable directory
PILOT_FILE = ""  # optional retained pilot.json from another session
SOURCE_COMMIT = "51e57b624ecd17fb3c1ba0c81998ce4ce4004292"
REL = "research/projects/cellular-language-models/experiments/006-path-first-clm/extension-parameter-depth/exploration-fast-001"
ROOT = Path("/kaggle/working/fast001-code-" + SOURCE_COMMIT[:12])
WORK = Path("/kaggle/working/fast001-output-" + SOURCE_COMMIT[:12])
if SOURCE_COMMIT.startswith("__"):
    raise RuntimeError("Import the committed Notebook with its pinned source")
if MODE not in ("preflight", "benchmark", "train-matrix", "report"):
    raise ValueError("Choose an explicit declared stage")
if MODE == "benchmark" and not ALLOW_GPU_PILOT:
    raise RuntimeError("Set ALLOW_GPU_PILOT=True only after reviewing the short pilot budget")
if MODE == "train-matrix" and not APPROVE_DEVELOPMENT:
    raise RuntimeError("Review pilot first, then explicitly set APPROVE_DEVELOPMENT=True")
ROOT.mkdir(parents=True, exist_ok=True)


def git(*args):
    print("[Setup] " + " ".join(args), flush=True)
    subprocess.run(["git", *args], cwd=ROOT, check=True)


if not (ROOT / ".git").exists():
    git("init", "-b", "fast001")
    git("remote", "add", "origin", "https://github.com/ArcheLabs/mini-cells.git")
    git("sparse-checkout", "init", "--no-cone")
    git(
        "sparse-checkout",
        "set",
        "--no-cone",
        "/pyproject.toml",
        "/src/",
        "/shared/python/",
        "/research/projects/cellular-language-models/experiments/006-path-first-clm/",
        "/research/projects/cellular-language-models/experiments/005-compositional-cell-graph/implementation/",
        "/research/projects/cellular-language-models/experiments/005-compositional-cell-graph/config/registered.json",
        "/research/governance/formal_seed_registry.json",
    )
    git("fetch", "--depth=1", "--filter=blob:none", "origin", SOURCE_COMMIT)
    git("checkout", "--detach", "FETCH_HEAD")
if subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=ROOT, text=True).strip() != SOURCE_COMMIT:
    raise RuntimeError("Existing checkout source differs; preserve previous Output")
ENTRY = ROOT / REL / "run_fast001.py"
WORK.mkdir(parents=True, exist_ok=True)
ENV = {**os.environ, "OMP_NUM_THREADS": "1", "MKL_NUM_THREADS": "1", "PYTHONUNBUFFERED": "1"}
if MODE in ("preflight", "report"):
    ENV["CUDA_VISIBLE_DEVICES"] = ""


def run(*args):
    subprocess.run([sys.executable, str(ENTRY), *args], cwd=ROOT, env=ENV, check=True)

In [ ]:
# Install data dependencies only for the explicitly approved development stage.
if MODE == "train-matrix":
    missing = [
        p for p in ("datasets", "tokenizers", "huggingface_hub") if importlib.util.find_spec(p) is None
    ]
    if missing:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "--disable-pip-version-check", *missing], check=True
        )
    # Optional HF credential uses the established Kaggle secret label; public data also works anonymously.
    if not ENV.get("HF_TOKEN"):
        try:
            from kaggle_secrets import UserSecretsClient

            ENV["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN").strip()
        except Exception:
            print("HF_TOKEN unavailable; public dataset access will be anonymous", flush=True)

if MODE == "preflight":
    run("--mode", "preflight", "--output", str(WORK / "cpu-gate"))
elif MODE == "benchmark":
    run("--mode", "benchmark", "--device", "cuda:0", "--allow-gpu-pilot", "--output", str(WORK / "pilot"))
    print((WORK / "pilot/pilot.json").read_text())
    print("Pilot only. Review projected_total_seconds; training has not started.")
elif MODE == "train-matrix":
    run("--mode", "preflight", "--output", str(WORK / "cpu-gate"))
    config = WORK / "frozen-development.json"
    if RESTORE_FROM:
        if not RESUME:
            raise RuntimeError("For retained checkpoints set RESUME=True; never overwrite them")
        run("--mode", "restore", "--bundle", RESTORE_FROM, "--output", str(WORK / "development"))
        config = WORK / "development/execution-config.json"
    elif not config.exists():
        pilot = Path(PILOT_FILE) if PILOT_FILE else WORK / "pilot/pilot.json"
        args = ["--mode", "freeze", "--pilot", str(pilot), "--approve-development", "--output", str(config)]
        if ACCEPT_SLOWER_THAN_TARGET:
            args += ["--accept-slower"]
        run(*args)
    args = [
        "--mode",
        "train-matrix",
        "--config",
        str(config),
        "--devices",
        "cuda:0",
        "cuda:1",
        "--arms",
        "CLM-S16",
        "CLM-S8",
        "M0-TINY",
        "--output",
        str(WORK / "development"),
    ]
    if RESUME:
        args += ["--resume"]
    run(*args)
else:
    config = WORK / "development/execution-config.json"
    run("--mode", "report", "--config", str(config), "--work", str(WORK / "development"))
print("Retain this Kaggle Output with Save Version:", WORK)
summary=WORK/'notebook-timing.json'
summary.write_text(json.dumps({'mode':MODE,'source_sha':SOURCE_COMMIT,
    'end_to_end_seconds':time.perf_counter()-SESSION_STARTED,'scientific_decision':False},indent=2)+'\n')


## Retention and limits

No hard runtime limit or automatic kill is installed. An optional CLI `--max-session-seconds` performs a checkpoint pause after a completed optimizer step only; it is not enabled here. Exported portable Output contains source/config/data/checkpoint hashes and RNG state; weights and raw tokens never belong in Git. Attach retained Output in a new cloud session before restoring it. Local `/working` alone is not persistence.

The two CLM workers start first. Whichever finishes first takes M0-TINY. Offline evaluate/profile/intervene/report begin only after all training workers have exited. Performance comparisons stay serial on one idle T4. All results are development diagnostics, never independent confirmation or CLM scientific superiority.
